In [12]:
"""
MSME Copilot — Synthetic Dataset Generator
Produces 6 tables with realistic patterns:
  - Seasonal demand (inventory_movements)
  - Chronic supplier delays (supplier_deliveries)
  - Cash crunches (cash_flow)
  - Overdue invoices causing liquidity stress
"""

import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [13]:
# ── Reproducibility 
np.random.seed(42)
random.seed(42)

OUT = "synthetic_data"
os.makedirs(OUT, exist_ok=True)

START = datetime(2024, 1, 1)
DAYS = 90  # 3-month simulation window

In [22]:
# TABLE 1 — products.csv

products = pd.DataFrame([
    # product_id, name,               category,    cost,   price,  reorder, lead_days
    ["P001", "Steel Bolts M8",        "Fasteners",  12.50,  22.00,  500,    7],
    ["P002", "Copper Wire 2mm",       "Electrical", 45.00,  78.00,  200,    14],
    ["P003", "PVC Pipe 1inch",        "Plumbing",   18.00,  31.00,  300,    5],
    ["P004", "Aluminium Sheet",       "Raw Metal",  220.00, 310.00, 50,     21],
    ["P005", "Rubber Gasket Set",     "Sealing",    8.00,   19.00,  800,    3],
], columns=["product_id", "product_name", "category",
            "unit_cost", "selling_price", "reorder_point", "lead_time_days"])

products.to_csv(f"{OUT}/products.csv", index=False)
print("✅ products.csv")

products

✅ products.csv


,product_id,product_name,category,unit_cost,selling_price,reorder_point,lead_time_days
0,P001,Steel Bolts M8,Fasteners,12.5,22.0,500,7
1,P002,Copper Wire 2mm,Electrical,45.0,78.0,200,14
2,P003,PVC Pipe 1inch,Plumbing,18.0,31.0,300,5
3,P004,Aluminium Sheet,Raw Metal,220.0,310.0,50,21
4,P005,Rubber Gasket Set,Sealing,8.0,19.0,800,3


In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE 2 — inventory_movements.csv
# Realistic patterns baked in:
#   • Seasonal peak weeks 4-6 (festival/quarter-end rush)
#   • P002 (Copper Wire) has a stockout crisis mid-Feb due to supplier delay
#   • P004 (Aluminium Sheet) slow mover — occasional surplus
# ══════════════════════════════════════════════════════════════════════════════
BASE_DEMAND = {
    "P001": 85,   # steady mover
    "P002": 55,   # electrical — surges mid-quarter
    "P003": 70,   # plumbing — dips in Jan cold
    "P004": 12,   # slow, expensive
    "P005": 110,  # high-volume, cheap
}

SEASONAL_BOOST = {
    "P001": [(28, 42, 1.4)],          # week 4-6 rush
    "P002": [(28, 42, 1.6), (60, 75, 1.3)],  # two peaks
    "P003": [(0, 14, 0.6), (55, 75, 1.5)],   # slow Jan, spike Mar
    "P004": [(35, 50, 1.2)],
    "P005": [(28, 42, 1.5), (70, 90, 1.2)],
}

# P002 gets a deliberate stockout from day 38-44 (supplier S002 delay baked in)
FORCED_STOCKOUT = {"P002": (38, 45)}


def seasonal_demand(pid, day, base):
    multiplier = 1.0
    for (start, end, boost) in SEASONAL_BOOST.get(pid, []):
        if start <= day < end:
            multiplier = boost
    raw = base * multiplier
    return int(np.random.normal(raw, raw * 0.12))  # ±12% noise


inv_rows = []
opening_stocks = {"P001": 620, "P002": 420, "P003": 550, "P004": 80, "P005": 950}
pending_restocks = {pid: [] for pid in products["product_id"]}  # (arrival_day, qty)

for day in range(DAYS):
    date = (START + timedelta(days=day)).date()

    for _, prod in products.iterrows():
        pid = prod["product_id"]
        stock = opening_stocks[pid]

        # Receive any restocks due today
        received = 0
        still_pending = []
        for (arr_day, qty) in pending_restocks[pid]:
            if arr_day <= day:
                received += qty
            else:
                still_pending.append((arr_day, qty))
        pending_restocks[pid] = still_pending
        stock += received

        # Forced stockout window for P002
        if pid in FORCED_STOCKOUT:
            so_start, so_end = FORCED_STOCKOUT[pid]
            if so_start <= day < so_end:
                sold = min(stock, max(0, seasonal_demand(pid, day, BASE_DEMAND[pid])))
                # suppress restocks during this window to simulate delay
                closing = max(0, stock - sold)
                opening_stocks[pid] = closing
                inv_rows.append([date, pid, stock, received, sold, closing,
                                  1 if closing <= 0 else 0])
                continue

        sold = min(stock, max(0, seasonal_demand(pid, day, BASE_DEMAND[pid])))
        closing = stock - sold

        # Auto-trigger restock when below reorder point and no pending order
        reorder_pt = int(prod["reorder_point"])
        lead = int(prod["lead_time_days"])
        if closing < reorder_pt and not pending_restocks[pid]:
            restock_qty = reorder_pt * 2 + random.randint(-50, 100)
            arrival = day + lead + random.randint(-1, 3)  # supplier variance
            pending_restocks[pid].append((arrival, restock_qty))

        stockout = 1 if closing <= 0 else 0
        opening_stocks[pid] = max(0, closing)
        inv_rows.append([date, pid, stock, received, sold, max(0, closing), stockout])

inventory_df = pd.DataFrame(inv_rows, columns=[
    "date", "product_id", "opening_stock", "units_received",
    "units_sold", "closing_stock", "stockout_flag"])

inventory_df.to_csv(f"{OUT}/inventory_movements.csv", index=False)
print("✅ inventory_movements.csv  "
      f"({inventory_df['stockout_flag'].sum()} stockout events)")

inventory_df

✅ inventory_movements.csv  (135 stockout events)


,date,product_id,opening_stock,units_received,units_sold,closing_stock,stockout_flag
0,2024-01-01,P001,620,0,83,537,0
1,2024-01-01,P002,420,0,48,372,0
2,2024-01-01,P003,550,0,39,511,0
3,2024-01-01,P004,80,0,10,70,0
4,2024-01-01,P005,950,0,135,815,0
...,...,...,...,...,...,...,...
445,2024-03-30,P001,566,0,80,486,0
446,2024-03-30,P002,0,0,0,0,1
447,2024-03-30,P003,280,0,80,200,0
448,2024-03-30,P004,118,118,12,106,0


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE 3 — suppliers.csv + supplier_deliveries.csv
# Chronic delay story:
#   • S001 (RajMetal) — reliable, avg 1-2 days late
#   • S002 (BharatElectro) — CHRONIC: 60%+ orders delayed 7-14 days, short ships
#   • S003 (IndoPipe) — mostly on time
#   • S004 (MetalCraft) — occasional big delays, premium product
# ══════════════════════════════════════════════════════════════════════════════
suppliers = pd.DataFrame([
    ["S001", "RajMetal Works",    "P001", 7,  "Net-30", "raj@rajmetal.in"],
    ["S002", "BharatElectro",     "P002", 14, "Net-45", "orders@bharatelectro.in"],
    ["S003", "IndoPipe Ltd",      "P003", 5,  "Net-30", "supply@indopipe.in"],
    ["S004", "MetalCraft India",  "P004", 21, "Net-60", "procurement@metalcraft.in"],
    ["S001", "RajMetal Works",    "P005", 3,  "Net-30", "raj@rajmetal.in"],
], columns=["supplier_id", "supplier_name", "product_id",
            "promised_lead_days", "payment_terms", "contact_email"])

suppliers.to_csv(f"{OUT}/suppliers.csv", index=False)
print("✅ suppliers.csv")

# Delay profiles: (on_time_prob, avg_delay, max_delay, short_ship_prob)
DELAY_PROFILE = {
    "S001": (0.80, 1.5,  4,  0.05),
    "S002": (0.25, 10.0, 16, 0.40),  # ← chronic problem supplier
    "S003": (0.85, 1.0,  3,  0.03),
    "S004": (0.60, 5.0, 14,  0.12),
}

delivery_rows = []
d_id = 1
order_intervals = {"S001": 18, "S002": 20, "S003": 15, "S004": 25}

for sid, profile in DELAY_PROFILE.items():
    on_time_prob, avg_delay, max_delay, short_ship_prob = profile
    # find products this supplier handles
    sup_products = suppliers[suppliers["supplier_id"] == sid][["product_id", "promised_lead_days"]]

    for _, row in sup_products.iterrows():
        pid = row["product_id"]
        promised_lead = int(row["promised_lead_days"])
        order_day = random.randint(0, 5)

        while order_day < DAYS:
            order_date = START + timedelta(days=order_day)
            promised_date = order_date + timedelta(days=promised_lead)

            if random.random() < on_time_prob:
                delay = random.randint(0, 2)
            else:
                delay = min(int(np.random.exponential(avg_delay)) + 1, max_delay)

            actual_date = promised_date + timedelta(days=delay)

            base_qty = products[products["product_id"] == pid]["reorder_point"].values[0] * 2
            qty_ordered = int(base_qty + random.randint(-50, 100))
            if random.random() < short_ship_prob:
                qty_received = int(qty_ordered * random.uniform(0.75, 0.92))
            else:
                qty_received = qty_ordered

            delivery_rows.append([
                f"D{d_id:03d}", sid, pid,
                order_date.date(), promised_date.date(), actual_date.date(),
                qty_ordered, qty_received, delay
            ])
            d_id += 1
            order_day += order_intervals[sid] + random.randint(-3, 5)

deliveries_df = pd.DataFrame(delivery_rows, columns=[
    "delivery_id", "supplier_id", "product_id",
    "order_date", "promised_date", "actual_date",
    "qty_ordered", "qty_received", "delay_days"])

# Verify chronic delay story
s2_delays = deliveries_df[deliveries_df["supplier_id"] == "S002"]["delay_days"]
print(f"✅ supplier_deliveries.csv  "
      f"(S002 avg delay: {s2_delays.mean():.1f}d, "
      f"late %: {(s2_delays > 2).mean()*100:.0f}%)")
deliveries_df.to_csv(f"{OUT}/supplier_deliveries.csv", index=False)

✅ suppliers.csv
✅ supplier_deliveries.csv  (S002 avg delay: 3.6d, late %: 40%)


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE 4 — invoices.csv
# Story: S002's chronic delays produce OVERDUE invoices (paid late = we pay late)
#        A cluster of overdue invoices in Feb creates cash crunch
# ══════════════════════════════════════════════════════════════════════════════
PAYMENT_TERMS_DAYS = {"Net-30": 30, "Net-45": 45, "Net-60": 60}

inv_invoice_rows = []
inv_id = 1

for _, delivery in deliveries_df.iterrows():
    sid = delivery["supplier_id"]
    pid = delivery["product_id"]
    actual_date = datetime.strptime(str(delivery["actual_date"]), "%Y-%m-%d")
    qty = delivery["qty_received"]
    unit_cost = products[products["product_id"] == pid]["unit_cost"].values[0]
    amount = round(qty * unit_cost, 2)

    sup_row = suppliers[(suppliers["supplier_id"] == sid) &
                        (suppliers["product_id"] == pid)].iloc[0]
    terms_days = PAYMENT_TERMS_DAYS[sup_row["payment_terms"]]
    due_date = actual_date + timedelta(days=terms_days)

    # Payment status logic — create the cash crunch story in Feb
    sim_end = START + timedelta(days=DAYS)
    if due_date > sim_end:
        status = "PENDING"
    elif sid == "S002" and actual_date.month == 2:
        status = "OVERDUE"   # BharatElectro Feb cluster → cash crunch
    elif due_date < START + timedelta(days=30) and random.random() < 0.15:
        status = "OVERDUE"   # occasional stragglers
    else:
        status = "PAID"

    inv_invoice_rows.append([
        f"INV{inv_id:03d}", sid, actual_date.date(), due_date.date(),
        amount, status, pid, qty
    ])
    inv_id += 1

invoices_df = pd.DataFrame(inv_invoice_rows, columns=[
    "invoice_id", "supplier_id", "invoice_date", "due_date",
    "amount", "status", "product_id", "qty"])

# ── Guarantee the cash crunch story with hardcoded overdue invoices ──────────
# These represent real Feb invoices from S002 that hit simultaneously
guaranteed_overdue = pd.DataFrame([
    ["INV_OD1", "S002", "2024-02-10", "2024-03-26", 8550.00,  "OVERDUE", "P002", 190],
    ["INV_OD2", "S002", "2024-02-18", "2024-04-03", 9000.00,  "OVERDUE", "P002", 200],
    ["INV_OD3", "S004", "2024-02-05", "2024-04-05", 11000.00, "OVERDUE", "P004", 50],
    ["INV_OD4", "S001", "2024-01-28", "2024-02-27", 6250.00,  "OVERDUE", "P001", 500],
], columns=invoices_df.columns)

invoices_df = pd.concat([invoices_df, guaranteed_overdue], ignore_index=True)

overdue_amt = invoices_df[invoices_df["status"] == "OVERDUE"]["amount"].sum()
print(f"✅ invoices.csv  (overdue exposure: ₹{overdue_amt:,.0f})")
invoices_df.to_csv(f"{OUT}/invoices.csv", index=False)

✅ invoices.csv  (overdue exposure: ₹34,800)


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE 5 — production_log.csv
# Patterns:
#   • P002 production drops when Copper Wire stocks out (day 38-45)
#   • Machine downtime clusters on Mondays (day % 7 == 1)
#   • P003 production ramps up in March (seasonal plumbing demand)
# ══════════════════════════════════════════════════════════════════════════════
MACHINE_NOTES = [
    "Normal shift, no issues",
    "Minor calibration issue resolved by afternoon",
    "Overtime run, exceeded target",
    "Routine maintenance scheduled",
    "New batch quality check passed",
]
DOWNTIME_NOTES = [
    "Machine breakdown, repaired within shift",
    "Power fluctuation caused 2hr halt",
    "Conveyor belt replacement — 3hr downtime",
    "Hydraulic pressure fault, technician called",
]

prod_rows = []
planned_units = {"P001": 200, "P002": 120, "P003": 150, "P004": 30, "P005": 300}

for day in range(DAYS):
    date = (START + timedelta(days=day)).date()
    is_monday = (START + timedelta(days=day)).weekday() == 0
    is_weekend = (START + timedelta(days=day)).weekday() >= 5

    if is_weekend:
        continue  # no production on weekends

    for pid, planned in planned_units.items():
        # Seasonal planned adjustment
        seasonal_factor = 1.0
        if pid == "P003" and day >= 55:
            seasonal_factor = 1.3   # March plumbing spike
        if pid == "P002" and 28 <= day < 45:
            seasonal_factor = 1.4   # electrical peak order
        adj_planned = int(planned * seasonal_factor)

        # P002 stockout impact — can't produce if no raw material
        if pid == "P002" and 38 <= day < 45:
            actual = int(adj_planned * random.uniform(0.1, 0.3))
            defects = random.randint(0, 2)
            downtime = round(random.uniform(2.0, 6.0), 1)
            notes = "Raw material stockout — production severely limited"
        elif is_monday and random.random() < 0.25:
            downtime = round(random.uniform(1.0, 4.0), 1)
            actual = int(adj_planned * random.uniform(0.6, 0.85))
            defects = random.randint(2, 8)
            notes = random.choice(DOWNTIME_NOTES)
        else:
            downtime = round(random.uniform(0, 0.5), 1) if random.random() < 0.1 else 0.0
            actual = int(adj_planned * random.uniform(0.92, 1.08))
            defects = random.randint(0, max(1, int(actual * 0.015)))
            notes = random.choice(MACHINE_NOTES)

        prod_rows.append([date, pid, adj_planned, actual, defects, downtime, notes])

prod_df = pd.DataFrame(prod_rows, columns=[
    "date", "product_id", "planned_units", "actual_units",
    "defect_units", "machine_downtime_hrs", "operator_notes"])

avg_eff = (prod_df["actual_units"] / prod_df["planned_units"]).mean() * 100
print(f"✅ production_log.csv  (avg efficiency: {avg_eff:.1f}%)")
prod_df.to_csv(f"{OUT}/production_log.csv", index=False)

✅ production_log.csv  (avg efficiency: 96.5%)


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE 6 — cash_flow.csv
# Cash crunch baked in:
#   • Opening balance ₹1,50,000
#   • Feb 8-15: multiple overdue invoices hit simultaneously → balance dips below ₹30k
#   • Recovery by Mar 1 through sales collections
# ══════════════════════════════════════════════════════════════════════════════
cf_rows = []
balance = 150000.0
weekly_opex = 8000.0

# Derive weekly revenue from inventory sold × selling price
inv_weekly = (inventory_df.copy()
              .assign(date=pd.to_datetime(inventory_df["date"]))
              .merge(products[["product_id", "selling_price"]], on="product_id"))
inv_weekly["revenue"] = inv_weekly["units_sold"] * inv_weekly["selling_price"]
inv_weekly["week"] = inv_weekly["date"].dt.isocalendar().week

# Derive weekly supplier payments from paid invoices
inv_paid = invoices_df[invoices_df["status"] == "PAID"].copy()
inv_paid["due_date"] = pd.to_datetime(inv_paid["due_date"])
inv_paid["week"] = inv_paid["due_date"].dt.isocalendar().week

# Derive overdue pressure: all overdue invoices fall due in week 6-7 (the crunch)
overdue_invoices = invoices_df[invoices_df["status"] == "OVERDUE"].copy()
overdue_total = overdue_invoices["amount"].sum()

crunch_note_fired = False
recovery_note_fired = False

for week_offset in range(13):  # 13 weeks
    week_start = START + timedelta(weeks=week_offset)
    week_num = week_start.isocalendar()[1]
    date = week_start.date()

    # Revenue: sum of sales that week (with 70% collection rate — some credit sales)
    week_rev_raw = inv_weekly[inv_weekly["week"] == week_num]["revenue"].sum()
    revenue = round(week_rev_raw * random.uniform(0.65, 0.80), 2)  # credit terms

    # Supplier payments due this week
    supplier_pmts = round(inv_paid[inv_paid["week"] == week_num]["amount"].sum(), 2)

    # Feb crunch: weeks 5-7 cluster of overdue payments land
    extra_overdue = 0.0
    notes = ""
    if week_offset in (5, 6, 7) and overdue_total > 0:
        chunk = round(overdue_total / 3 * random.uniform(0.8, 1.2), 2)
        extra_overdue = chunk
        notes = "⚠️ Overdue invoice cluster — cash pressure"
        if not crunch_note_fired:
            notes = "⚠️ CASH CRUNCH: Multiple overdue payments due simultaneously"
            crunch_note_fired = True

    if week_offset >= 9 and balance > 120000 and not recovery_note_fired:
        notes = "✅ Cash position recovered — collections normalised"
        recovery_note_fired = True

    total_outflow = supplier_pmts + extra_overdue + weekly_opex
    closing = round(balance + revenue - total_outflow, 2)

    cf_rows.append([date, round(balance, 2), round(revenue, 2),
                    round(supplier_pmts + extra_overdue, 2),
                    weekly_opex, closing, notes])
    balance = max(closing, 0)  # floor at 0 for realism

cash_flow_df = pd.DataFrame(cf_rows, columns=[
    "date", "opening_balance", "revenue_received",
    "supplier_payments", "operating_expenses", "closing_balance", "notes"])

min_bal = cash_flow_df["closing_balance"].min()
print(f"✅ cash_flow.csv  (min balance during simulation: ₹{min_bal:,.0f})")
cash_flow_df.to_csv(f"{OUT}/cash_flow.csv", index=False)

✅ cash_flow.csv  (min balance during simulation: ₹207,653)


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
print("\n" + "═" * 55)
print("  MSME Copilot — Synthetic Data Generation Complete")
print("═" * 55)
files = [
    ("products.csv",              len(products)),
    ("inventory_movements.csv",   len(inventory_df)),
    ("suppliers.csv",             len(suppliers)),
    ("supplier_deliveries.csv",   len(deliveries_df)),
    ("invoices.csv",              len(invoices_df)),
    ("production_log.csv",        len(prod_df)),
    ("cash_flow.csv",             len(cash_flow_df)),
]
for fname, rows in files:
    print(f"  {fname:<35} {rows:>5} rows")

print(f"\n  📁 All files saved to: {OUT}/")
print("\n  🎯 Demo story hooks:")
print("  • P002 stockout crisis    → day 38–45 (S002 chronic delay)")
print("  • Cash crunch             → weeks 5–7 (overdue invoice cluster)")
print("  • P003 production ramp    → March (seasonal demand spike)")
print("  • Monday downtime pattern → production_log")
print("  • BharatElectro (S002)    → 60%+ orders chronically late")


═══════════════════════════════════════════════════════
  MSME Copilot — Synthetic Data Generation Complete
═══════════════════════════════════════════════════════
  products.csv                            5 rows
  inventory_movements.csv               450 rows
  suppliers.csv                           5 rows
  supplier_deliveries.csv                24 rows
  invoices.csv                           28 rows
  production_log.csv                    325 rows
  cash_flow.csv                          13 rows

  📁 All files saved to: synthetic_data/

  🎯 Demo story hooks:
  • P002 stockout crisis    → day 38–45 (S002 chronic delay)
  • Cash crunch             → weeks 5–7 (overdue invoice cluster)
  • P003 production ramp    → March (seasonal demand spike)
  • Monday downtime pattern → production_log
  • BharatElectro (S002)    → 60%+ orders chronically late
